# AMASS BMLmovi: SMPL Ray-Traced Radar Point Clouds

Full pipeline: AMASS motion capture (SMPL pose/shape/trans) -> Mitsuba ray tracing -> FMCW radar -> 30-frame point cloud sequence.

Requires:
- BMLmovi data in `data/BMLmovi_full/` (extracted from `BMLmovi.tar.bz2`)
- SMPL model files in `models/smpl_models/`

In [ ]:
import sys, pathlib

repo_root = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / 'witwin-radar'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from core import Radar, Renderer, Scene
from sigproc import process_pc, process_rd

torch.set_default_device('cuda')
MODEL_ROOT = str(repo_root / 'models' / 'smpl_models')

## 1. Load AMASS BMLmovi Sequence

In [ ]:
data_dir = pathlib.Path.cwd().parent / 'data' / 'BMLmovi_full' / 'BMLmovi'
subjects = sorted(data_dir.iterdir())
print(f'{len(subjects)} subjects')

subj_dir = subjects[0]
pose_files = sorted(subj_dir.glob('*_poses.npz'))
shape_file = subj_dir / 'shape.npz'

pose_data = np.load(str(pose_files[0]))
shape_data = np.load(str(shape_file))

all_poses = pose_data['poses'][:, :72]   # SMPL body pose (drop hand params)
all_trans = pose_data['trans']            # global translation
betas = shape_data['betas'][:10]         # body shape
gender = str(shape_data['gender'])
source_fps = float(pose_data['mocap_framerate'])

print(f'Sequence: {pose_files[0].name}')
print(f'  {all_poses.shape[0]} frames at {source_fps}fps ({all_poses.shape[0]/source_fps:.1f}s)')
print(f'  Gender: {gender}, Betas: {betas[:5].round(2)}...')
print(f'  Keys: {list(pose_data.keys())}')

## 2. Radar Config

In [ ]:
config = {
    "num_tx": 3, "num_rx": 4, "fc": 77e9, "slope": 60.012,
    "adc_samples": 256, "adc_start_time": 6, "sample_rate": 4400,
    "idle_time": 7, "ramp_end_time": 65, "chirp_per_frame": 128,
    "frame_per_second": 10, "num_doppler_bins": 128, "num_range_bins": 256,
    "num_angle_bins": 64, "power": 15,
    "tx_loc": [[0, 0, 0], [4, 0, 0], [2, 1, 0]],
    "rx_loc": [[-6, 0, 0], [-5, 0, 0], [-4, 0, 0], [-3, 0, 0]],
}
radar = Radar(config)
print(f'Range res: {radar.range_resolution:.4f} m, Doppler res: {radar.doppler_resolution:.4f} m/s')

## 3. Scene Setup and Frame Selection

In [ ]:
step = int(source_fps / config['frame_per_second'])  # 12
num_frames = 30
start = 50
indices = [start + i * step for i in range(num_frames + 1)]

# Place body at Z=-3 in front of radar
t_ref = all_trans[indices[0]]
offset = np.array([-t_ref[0], -t_ref[1], -t_ref[2] - 3.0])

scene = Scene(fov=80)
scene.set_sensor(origin=(0, 0, 0), target=(0, 0, -5))

t0 = all_trans[indices[0]] + offset
scene.add_smpl('human', all_poses[indices[0]], betas,
               translation=t0.tolist(), gender=gender, model_root=MODEL_ROOT)
renderer = Renderer(scene, resolution=192)

# Quick preview of frame 0
pts0, int0 = renderer.trace()
print(f'Frame 0 ray trace: {pts0.shape[0]} reflection points')

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
p = pts0.cpu().numpy()
ax.scatter(p[:, 0], -p[:, 2], p[:, 1], c=int0.cpu().numpy(), cmap='hot', s=2)
ax.set_xlabel('X'); ax.set_ylabel('Depth'); ax.set_zlabel('Y')
ax.set_title(f'SMPL ray-traced ({pts0.shape[0]} pts)')
ax.set_box_aspect([1, 1.5, 1])
ax.view_init(elev=15, azim=-60)

## 4. Generate 30 Radar Frames

In [ ]:
T_chirp = (config['idle_time'] + config['ramp_end_time']) * 1e-6
T_frame = T_chirp * config['num_tx'] * config['chirp_per_frame']

all_pcs = []
all_rd_mags = []
all_ray_pts = []
ranges = radar.ranges.cpu().numpy()
velocities = radar.velocities.cpu().numpy()

for fi in range(num_frames):
    src_cur, src_next = indices[fi], indices[fi + 1]

    t_cur = all_trans[src_cur] + offset
    scene.update_smpl('human', all_poses[src_cur], betas, translation=t_cur.tolist())
    renderer = Renderer(scene, resolution=192)
    pts_cur, int_cur = renderer.trace()

    t_next = all_trans[src_next] + offset
    scene.update_smpl('human', all_poses[src_next], betas, translation=t_next.tolist())
    renderer_next = Renderer(scene, resolution=192)
    pts_next, int_next = renderer_next.trace()

    n = min(pts_cur.shape[0], pts_next.shape[0])
    if n == 0:
        all_pcs.append(np.zeros((0, 6)))
        all_rd_mags.append(np.zeros((128, 256)))
        all_ray_pts.append(np.zeros((0, 3)))
        continue

    p0, p1, inten = pts_cur[:n], pts_next[:n], int_cur[:n]

    def _interp(t, _p0=p0, _p1=p1, _i=inten):
        return _i, _p0 + (_p1 - _p0) * (t / T_frame)

    frame = radar.mimo(_interp, t0=0)
    pc = process_pc(radar, frame, static_clutter_removal=False, positive_velocity_only=False)
    rd_mag, _, _, _ = process_rd(radar, frame, tx=0, rx=0)

    all_pcs.append(pc)
    all_rd_mags.append(rd_mag)
    all_ray_pts.append(pts_cur.cpu().numpy())

    if fi % 5 == 0:
        print(f'  [{fi:2d}/{num_frames}] {pts_cur.shape[0]} ray pts -> {pc.shape[0]} PC pts')

print(f'Done. {sum(pc.shape[0] for pc in all_pcs)} total points')

## 5. Point Cloud Sequence

In [ ]:
fig = plt.figure(figsize=(20, 12))
show_idx = np.linspace(0, num_frames - 1, 6, dtype=int)

for i, idx in enumerate(show_idx):
    ax = fig.add_subplot(2, 3, i + 1, projection='3d')
    pc = all_pcs[idx]
    ray_pts = all_ray_pts[idx]

    if ray_pts.shape[0] > 0:
        ax.scatter(ray_pts[:, 0], -ray_pts[:, 2], ray_pts[:, 1],
                   c='lightblue', s=1, alpha=0.3)
    if pc.shape[0] > 0:
        ax.scatter(-pc[:, 0], pc[:, 1], pc[:, 2],
                   c=pc[:, 4], cmap='hot', s=8)

    ax.set_xlabel('X'); ax.set_ylabel('Depth'); ax.set_zlabel('Y')
    ax.set_title(f'Frame {idx} ({pc.shape[0]} pts)')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(1, 5); ax.set_zlim(-1.5, 1.5)
    ax.set_box_aspect([1, 1.5, 1])
    ax.view_init(elev=15, azim=-60)

plt.suptitle('AMASS BMLmovi: SMPL Ray-Traced Radar Point Clouds', fontsize=14)
plt.tight_layout()

## 6. Range-Doppler Maps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, idx in enumerate(show_idx):
    ax = axes[i // 3, i % 3]
    ax.imshow(all_rd_mags[idx][:, :len(ranges)],
              extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
              aspect='auto', origin='lower', cmap='jet')
    ax.set_xlabel('Range (m)'); ax.set_ylabel('Velocity (m/s)')
    ax.set_title(f'Frame {idx}')
plt.suptitle('Range-Doppler Maps (SMPL body)', fontsize=14)
plt.tight_layout()

## 7. 30-Frame Point Cloud Overlay

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
colors = plt.cm.viridis(np.linspace(0, 1, num_frames))

for i in range(num_frames):
    pc = all_pcs[i]
    if pc.shape[0] > 0:
        ax.scatter(-pc[:, 0], pc[:, 1], pc[:, 2], c=[colors[i]], s=3, alpha=0.3)

ax.set_xlabel('X (m)'); ax.set_ylabel('Depth (m)'); ax.set_zlabel('Y (m)')
ax.set_title('30-Frame Point Cloud Overlay (color = time)')
ax.set_xlim(-1.5, 1.5); ax.set_ylim(1, 5); ax.set_zlim(-1.5, 1.5)
ax.set_box_aspect([1, 1.5, 1])
ax.view_init(elev=15, azim=-60)
plt.tight_layout()